In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DistilBertTokenizer, DistilBertForSequenceClassification

In [ ]:
model_name2 = "nlptown/bert-base-multilingual-uncased-sentiment"
mymodel2 = AutoModelForSequenceClassification.from_pretrained(model_name2)
mytokenizer2 = AutoTokenizer.from_pretrained(model_name2)

classifier = pipeline("sentiment-analysis", model = mymodel2 , tokenizer = mytokenizer2)
res = classifier("I was so not happy with the Barbie Movie")
print(res)

In [ ]:
# Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example text
text = "I was so not happy with the Barbie Movie"

# Tokenize the text
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

In [ ]:
ids=tokenizer.convert_tokens_to_ids(tokens)
print("IDs:", ids)

In [ ]:

# Encode the text (tokenization + converting to input IDs)
encoded_input = tokenizer(text)
print("Encoded Input:", encoded_input)

In [ ]:
# Decode the text
decoded_output = tokenizer.decode(ids)
print("Decode Output: ", decoded_output)


In [ ]:
# Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Example text
text = "I was so not happy with the Barbie Movie"

# Tokenize the text
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

# Convert tokens to input IDs
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Input IDs:", input_ids)

# Encode the text (tokenization + converting to input IDs)
encoded_input = tokenizer(text)
print("Encoded Input:", encoded_input)

# Decode the text
decoded_output = tokenizer.decode(input_ids)
print("Decode Output: ", decoded_output)

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("/content/drive/MyDrive/Dataset/IMDB Dataset.csv")

In [ ]:
df


In [ ]:
df['review'] = df['review'].str.lower()

In [ ]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

In [ ]:
df['review'] = df['review'].apply(remove_html_tags)

In [ ]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

In [ ]:
df['review'] = df['review'].apply(remove_url)

In [ ]:
import string,time
string.punctuation

In [ ]:
exclude = string.punctuation
exclude

In [ ]:
def remove_punc(text):
    for char in exclude:
        text = text.replace(char,'')
    return text

In [ ]:
df['review'] = df['review'].apply(remove_punc)

In [ ]:
chat_words = {
    'AFAIK':'As Far As I Know',
    'AFK':'Away From Keyboard',
    'ASAP':'As Soon As Possible'
}


{
    "FYI": "For Your Information",
    "ASAP": "As Soon As Possible",
    "BRB": "Be Right Back",
    "BTW": "By The Way",
    "OMG": "Oh My God",
    "IMO": "In My Opinion",
    "LOL": "Laugh Out Loud",
    "TTYL": "Talk To You Later",
    "GTG": "Got To Go",
    "TTYT": "Talk To You Tomorrow",
    "IDK": "I Don't Know",
    "TMI": "Too Much Information",
    "IMHO": "In My Humble Opinion",
    "ICYMI": "In Case You Missed It",
    "AFAIK": "As Far As I Know",
    "BTW": "By The Way",
    "FAQ": "Frequently Asked Questions",
    "TGIF": "Thank God It's Friday",
    "FYA": "For Your Action",
    "ICYMI": "In Case You Missed It",
}

In [ ]:
def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)

In [ ]:
df['review'] = df['review'].apply(chat_conversion)

In [ ]:
import re
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [ ]:
df['review'] = df['review'].apply(remove_emoji)

In [ ]:
df['review'][0]

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# convert Pandas DataFrame (df) to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

#  tokenize function creat
def tokenize_function(examples):
    return tokenizer(examples['review'], padding="max_length", truncation=True)

# map call
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

In [ ]:
df.shape

In [ ]:
tokenized_datasets

In [ ]:
tokenized_datasets.to_pandas().head()

In [ ]:

def encode_sentiment(example):

    label = 1 if example['sentiment'] == 'positive' else 0


    return {'labels': label}


tokenized_datasets = tokenized_datasets.map(encode_sentiment)

tokenized_datasets = tokenized_datasets.remove_columns(["review", "sentiment"])


tokenized_datasets.to_pandas().head()

In [ ]:
from datasets import DatasetDict

dataset_split = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

print(dataset_split)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',          # Output directory
    eval_strategy ="epoch",     # Evaluate every epoch
    learning_rate=2e-5,              # Learning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    num_train_epochs=1,              # Number of training epochs
    weight_decay=0.01,               # Strength of weight decay
)
training_args

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
)

In [ ]:
trainer.train()